In [9]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
import pandas as pd

# 1. On détecte dynamiquement le dossier racine du projet
# Si le notebook est dans le dossier "notebooks", le dossier parent est la racine
PROJECT_DIR = Path.cwd().parent 

# 2. On construit les chemins de manière robuste (les "/" s'adaptent à l'OS)
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "all_leagues_merged_transformed.csv"
CLEAN_DATA_PATH = PROJECT_DIR / "data" / "clean" / "dataset_aggregated.csv"

# 3. Utilisation 
# df = pd.read_csv(RAW_DATA_PATH)
# df.to_csv(CLEAN_DATA_PATH, index=False)

In [15]:
import sys
import os
from pathlib import Path

# 1. Détection de Google Colab
if 'google.colab' in sys.modules:
    print("🌐 Environnement Colab détecté.")
    # On clone tout ton dépôt Github directement dans Colab pour avoir les données
    # /!\ Remplace l'URL par celle de TON vrai Github /!\
    !git clone https://github.com/Aymanyah/Stat_app.git
    
    # On définit le dossier projet comme étant le dossier qu'on vient de cloner
    PROJECT_DIR = Path('/content/Stat_app')

# 2. Détection Locale (Ta machine, Onyxia, VSCode, etc.)
else:
    print("💻 Environnement local détecté.")
    # On remonte d'un cran par rapport au notebook pour trouver la racine du projet
    PROJECT_DIR = Path.cwd().parent 

# Définition des dossiers
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_CLEAN = PROJECT_DIR / "data" / "clean"
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print(f"📂 Racine du projet définie sur : {PROJECT_DIR}")

💻 Environnement local détecté.
📂 Racine du projet définie sur : /home/onyxia/Stat_app


In [16]:
# Chargement depuis le dossier raw 
df = pd.read_csv('../data/raw/all_leagues_merged_transformed.csv')

1. Corriger les problèmes d'encodage sur les noms des joueurs et des clubs.

In [17]:
# Nettoyer les noms de colonnes
df.columns = [col.encode('latin1').decode('utf8') for col in df.columns]
df.columns = df.columns.str.strip()

# Fonction pour nettoyer les valeurs textuelles
def fix_encoding(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf8")
        except:
            return x
    return x

for col in ["player", "team", "league", "nation", "pos"]:
    if col in df.columns:
        df[col] = df[col].apply(fix_encoding)
        df[col] = df[col].str.strip()

# Supprimer les doublons exacts
df_clean = df.drop_duplicates()

# --- 1) COLONNES NUMÉRIQUES ---
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["season"]]

2. Agréger les statistiques des joueurs ayant changé de club en cours de saison.

In [18]:
# --- 2) DÉFINITION D'UNE FONCTION D'AGRÉGATION INTELLIGENTE ---
def smart_agg(col):
    if "percent" in col.lower() or "%" in col or "ratio" in col.lower():
        return "mean"
    if col in ["Minutes de jeu", "G", "G-PK", "Assists", "Tirs", "Tirs cadrés"]:
        return "sum"
    return "mean"

agg_dict = {col: smart_agg(col) for col in numeric_cols}

# --- 3) AJOUT DES COLONNES CATEGORIELLES ---
# On prend "last" pour team, league, nation, pos
categorical_cols = [c for c in ["team", "league", "nation", "pos"] if c in df_clean.columns]
for col in categorical_cols:
    agg_dict[col] = "last"

# --- 4) AGGRÉGATION PAR JOUEUR + SAISON ---
df_grouped = df_clean.groupby(["player", "season"]).agg(agg_dict).reset_index()

# --- 5) PIVOT DES STATISTIQUES NUMÉRIQUES ---
df_stats = df_grouped.pivot(index="player", columns="season", values=numeric_cols)
df_stats.columns = [f"{stat}_{season}" for stat, season in df_stats.columns]
df_stats = df_stats.reset_index()

# --- 6) PIVOT DES COLONNES CATEGORIELLES ---
df_cats = {}
for col in categorical_cols:
    df_cat = df_grouped.pivot(index="player", columns="season", values=col)
    df_cat.columns = [f"{col}_{season}" for season in df_cat.columns]
    df_cats[col] = df_cat.reset_index()

# --- 7) FUSION DE TOUTES LES COLONNES ---
df_final = df_stats
for col, df_cat in df_cats.items():
    df_final = df_final.merge(df_cat, on="player", how="left")

3.Sauvegarder un jeu de données propre pour l'analyse exploratoire et la modélisation.

In [19]:
# --- 8) SUPPRESSION INTELLIGENTE DES COLONNES 2021 POUR LES STATS NUMÉRIQUES ---
for col in df_final.columns:
    if col.endswith("_2020.0"):
        col_2021 = col.replace("_2020.0", "_2021.0")
        if col_2021 in df_final.columns:
            df_final[col] = df_final[col].fillna(df_final[col_2021])
            df_final = df_final.drop(columns=[col_2021])

# --- 9) GARDER UNE SEULE COLONNE BORN ET POS ---
born_cols = [c for c in df_final.columns if c.startswith("born_")]
if born_cols:
    df_final["born"] = df_final[born_cols].bfill(axis=1).iloc[:, 0]
    df_final = df_final.drop(columns=born_cols)

pos_cols = [c for c in df_final.columns if c.startswith("pos_")]
if pos_cols:
    df_final["pos"] = df_final[pos_cols].bfill(axis=1).iloc[:, 0]
    df_final = df_final.drop(columns=pos_cols)


# --- 10) METTRE EN % LES VALEURS EN POURCENTAGE ---
percent_cols = [c for c in df_final.columns if "%" in c]
for col in percent_cols:
    if df_final[col].mean(skipna=True) > 1:
        df_final[col] = df_final[col] / 100


df_final.head()

# --- 11) SAUVEGARDE DYNAMIQUE AVEC PATHLIB (METHODE 1) ---
from pathlib import Path

# On définit la racine du projet (On remonte d'un dossier depuis 'notebooks')
PROJECT_DIR = Path.cwd().parent 

# On crée le chemin dynamique vers le dossier data/clean/
DATA_CLEAN = PROJECT_DIR / "data" / "clean"

# Sécurité : on s'assure que le dossier 'clean' existe bien. 
# S'il n'existe pas, Python le crée automatiquement !
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

# On génère le chemin complet du fichier
fichier_sauvegarde = DATA_CLEAN / "dataset_aggregated.csv"

# Sauvegarde finale
df_final.to_csv(fichier_sauvegarde, index=False)

print(f"✅ Nettoyage terminé ! Fichier sauvegardé dynamiquement dans : {fichier_sauvegarde}")

✅ Nettoyage terminé ! Fichier sauvegardé dynamiquement dans : /home/onyxia/Stat_app/data/clean/dataset_aggregated.csv
